# 🛡️ CyberFlow IDS: ML-Powered Network Anomaly Detection - Model Tournament

Welcome to the **CyberFlow IDS** Model Tournament and Training pipeline. This notebook is designed to run seamlessly on **Google Colab** or locally. It automates the following end-to-end steps:
1. **Environment Setup**: Installs required libraries (including `xgboost`).
2. **Data Ingestion**: Downloads the **UNB CIC-IDS2017** dataset from a verified Google Drive high-speed mirror.
3. **Memory-Safe Downsampling**: Cleans raw packet features, removes infinities/NaNs, maps labels to binary threats, and downsamples traffic to a balanced 5% stratified subset.
4. **Auto-Leakage Shield**: Automatically detects and prunes proxy columns having a correlation coefficient $> 0.98$ with the target to ensure realistic training.
5. **Model Tournament**: Trains and compares **Decision Tree**, **Random Forest**, **Naive Bayes**, and **XGBoost**.
6. **Champion Export**: Saves the champion classifier and feature scaler for use in the real-time detection engine.

## 🛠️ Step 1: Environment Setup & Installations
We install the core libraries and set up our project folder structure (`data/raw`, `data/processed`, `models`, `reports`).

In [ ]:
# Install dependencies if not present
!pip install -q xgboost joblib scikit-learn matplotlib seaborn gdown pandas numpy

import os

# Create project directories
directories = [
    'data/raw',
    'data/processed',
    'models',
    'reports'
]
for d in directories:
    os.makedirs(d, exist_ok=True)
    print(f'Created/verified directory: {d}')

## 📦 Step 2: Download Raw Dataset
We connect to a high-speed Google Drive mirror hosting the **CIC-IDS-2017** Machine Learning CSV zip file, download it, and extract it to `data/raw`.

In [ ]:
import zipfile
import gdown

file_id = '1-t3RdDpmqMs4ABt9oobSapeNYTZJ9tpu'
zip_path = 'data/raw/MachineLearningCSV.zip'
target_dir = 'data/raw'

print('🚀 Downloading UNB CIC-IDS-2017 dataset from Google Drive...')
try:
    gdown.download(id=file_id, output=zip_path, quiet=False)
    print('📦 Download complete. Extracting CSV sheets...')
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
        
    os.remove(zip_path) # Clean up zip to save disk space
    
    # Verify extraction
    final_folder = os.path.join(target_dir, 'MachineLearningCVE')
    if os.path.exists(final_folder):
        print(f'\n🏆 SUCCESS! Files extracted to: {final_folder}')
        print('Files found:', os.listdir(final_folder))
    else:
        print(f'\n🏆 SUCCESS! Files extracted directly to: {target_dir}')
        print('Files found:', os.listdir(target_dir))
except Exception as e:
    print(f'❌ Download or extraction failed: {e}')

## 🔄 Step 3: Memory-Safe Downsampling & Cleansing
The raw dataset is large, so we process it in chunks. We clean the numeric features (handle infs/NaNs), map classes, and export a balanced 5% stratified sample for tournament training.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

raw_dir = 'data/raw'
output_file = 'data/processed/sampled_traffic.csv'

# Find extracted CSVs
csv_files = []
for root, dirs, files in os.walk(raw_dir):
    for file in files:
        if file.lower().endswith('.csv') and not file.startswith('.'):
            csv_files.append(os.path.join(root, file))

if not csv_files:
    print(f'❌ No CSV files found in {raw_dir}. Please upload or extract them first!')
else:
    target_csv = csv_files[0]
    print(f'🎯 Target raw dataset identified: {target_csv}')
    
    try:
        # Detect delimiter and clean BOM/encoding
        with open(target_csv, 'r', encoding='utf-8-sig', errors='ignore') as f:
            first_line = f.readline()
            
        separator = ';' if (';' in first_line and first_line.count(';') > first_line.count(',')) else ','
        print(f'ℹ️ Column separator detected: {separator}')
        
        # Check headers
        preview = pd.read_csv(target_csv, sep=separator, nrows=1, encoding='utf-8-sig')
        preview.columns = preview.columns.str.strip().str.replace('"', '').str.replace("'", '')
        
        possible_labels = ['label', 'class', 'category', 'threat', 'attack', 'target', 'type']
        detected_label_col = None
        for col in preview.columns:
            if any(term in str(col).lower() for term in possible_labels):
                detected_label_col = col
                break
                
        if detected_label_col:
            print(f'🎯 Classification column matched: {detected_label_col}')
        else:
            raise KeyError('No suitable label column detected. Headers found: ' + str(list(preview.columns)))

        print('\n🔄 Downsampling & cleaning in memory-safe chunks...')
        sampled_chunks = []
        chunk_size = 100000
        
        for i, chunk in enumerate(pd.read_csv(target_csv, sep=separator, chunksize=chunk_size, low_memory=False, encoding='utf-8-sig')):
            chunk.columns = chunk.columns.str.strip().str.replace('"', '').str.replace("'", '')
            chunk = chunk.rename(columns={detected_label_col: 'Label'})
            chunk = chunk.dropna(subset=['Label'])
            
            # Encode target
            chunk['Target'] = chunk['Label'].apply(lambda x: 0 if str(x).strip().upper() in ['BENIGN', '0', 'NORMAL'] else 1)
            
            numeric_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
            if 'Target' in numeric_cols:
                numeric_cols.remove('Target')
                
            chunk[numeric_cols] = chunk[numeric_cols].replace([np.inf, -np.inf], np.nan)
            chunk = chunk.dropna(subset=numeric_cols)
            
            if len(chunk) > 10:
                try:
                    _, sub_sample = train_test_split(chunk, test_size=0.05, stratify=chunk['Target'], random_state=42)
                    sampled_chunks.append(sub_sample)
                except ValueError:
                    sampled_chunks.append(chunk.sample(frac=0.05, random_state=42))
            
            if (i+1) % 5 == 0:
                print(f'   Processed chunk {i+1}...')

        final_sample_df = pd.concat(sampled_chunks, ignore_index=True)
        final_sample_df.to_csv(output_file, index=False)
        
        print(f'\n🏆 SUCCESS! Sampled dataset saved to: {output_file}')
        print(f'📊 Shape: {final_sample_df.shape[0]} rows, {final_sample_df.shape[1]} features')
        print(final_sample_df['Label'].value_counts())
    except Exception as e:
        print(f'❌ Ingestion failed: {e}')

## ⚔️ Step 4: Model Tournament (with XGBoost)
We run the model tournament matching our latest local training setup. This runs a diagnostic data leakage check, scales features, trains the roster (Decision Tree, Random Forest, XGBoost, Naive Bayes), crowns a champion, and exports the model binaries.

In [ ]:
import time
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import joblib

processed_file = 'data/processed/sampled_traffic.csv'
model_save_dir = 'models'

if not os.path.exists(processed_file):
    print(f'❌ Error: Processed data file not found at {processed_file}')
else:
    df = pd.read_csv(processed_file)
    
    # Standardize target mapping just in case
    df['Target'] = df['Label'].apply(lambda x: 0 if any(term in str(x).upper() for term in ['BENIGN', 'NORMAL', '0']) else 1)
    y = df['Target'].values
    
    metadata_cols = [
        'Label', 'Target', 'Flow ID', 'Source IP', 'Source Port', 
        'Destination IP', 'Destination Port', 'Timestamp', 'Protocol'
    ]
    features_df = df.drop(columns=[col for col in metadata_cols if col in df.columns])
    numeric_features = features_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Data Leakage Protection Shield
    print('🔍 Scanning feature space for proxy attributes...')
    leaky_features = []
    constant_features = []
    
    for col in list(numeric_features):
        std_val = df[col].std()
        if std_val == 0 or np.isnan(std_val):
            constant_features.append(col)
            if col in numeric_features:
                numeric_features.remove(col)
            continue
        correlation = df[col].corr(df['Target'])
        if abs(correlation) > 0.98:
            leaky_features.append((col, correlation))
            if col in numeric_features:
                numeric_features.remove(col)
                
    if leaky_features:
        print(f'🛡️ Leakage Shield Active: Excluded {len(leaky_features)} leaky columns: {[x[0] for x in leaky_features]}')
    else:
        print('✅ No direct linear leakage columns found.')
        
    X = features_df[numeric_features].values
    print(f'⚙️ Isolated {X.shape[1]} numeric features for training.')
    
    # Stratified Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    scaler_path = os.path.join(model_save_dir, 'scaler.joblib')
    joblib.dump(scaler, scaler_path)
    print(f'💾 Feature Scaler saved to {scaler_path}')
    
    # Model Tournament combat roster
    tournament_models = {
        'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_split=10, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=30, max_depth=8, random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=30, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'),
        'Naive Bayes': GaussianNB()
    }
    
    results = []
    print('\n⚔️ Model Tournament Combat Phase Starting...\n')
    print('-' * 85)
    print(f"{'Model Name':<18} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Train Time':<10}")
    print('-' * 85)
    
    best_f1 = 0.0
    best_model_name = ''
    best_model_instance = None
    
    for name, model in tournament_models.items():
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        elapsed_time = time.time() - start_time
        
        y_pred = model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
        
        results.append({
            'model': name,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'time': elapsed_time
        })
        print(f'{name:<18} | {accuracy:.4f}     | {precision:.4f}     | {recall:.4f}     | {f1:.4f}     | {elapsed_time:.2f}s')
        
        if f1 > best_f1:
            best_f1 = f1
            best_model_name = name
            best_model_instance = model
            
    print('-' * 85)
    champion_path = os.path.join(model_save_dir, 'intrusion_detector.joblib')
    joblib.dump(best_model_instance, champion_path)
    
    print(f'\n🏆 CHAMPION CROWNED: {best_model_name.upper()} (F1-Score: {best_f1:.4f})')
    print(f'💾 Saved model weights to: {champion_path}')

## 📈 Step 5: Visualizing Tournament Results
We plot and display the evaluation metrics for all classifiers side-by-side.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_path = 'reports/model_comparison.png'

print('📊 Rendering metric comparison graphs...')
results_df = pd.DataFrame(results)
melted_df = pd.melt(
    results_df, 
    id_vars=['model'], 
    value_vars=['accuracy', 'precision', 'recall', 'f1'],
    var_name='Metric', 
    value_name='Score'
)

metric_map = {
    'accuracy': 'Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-Score'
}
melted_df['Metric'] = melted_df['Metric'].map(metric_map)

sns.set_theme(style='whitegrid')
plt.figure(figsize=(11, 6), dpi=150)

ax = sns.barplot(
    x='Metric', 
    y='Score', 
    hue='model', 
    data=melted_df, 
    palette='viridis'
)

plt.title('CyberFlow IDS Model Tournament - Performance Comparison', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Evaluation Metrics', fontsize=11, fontweight='bold', labelpad=10)
plt.ylabel('Score (0.0 to 1.0)', fontsize=11, fontweight='bold', labelpad=10)
plt.ylim(0.0, 1.1)
plt.legend(title='Machine Learning Classifiers', loc='lower left', frameon=True)

for p in ax.patches:
    val = p.get_height()
    if val > 0:
        ax.annotate(
            f'{val:.3f}',
            (p.get_x() + p.get_width() / 2., val),
            ha='center', va='center',
            xytext=(0, 6),
            textcoords='offset points',
            fontsize=8,
            fontweight='bold',
            color='#333333'
        )

plt.tight_layout()
plt.savefig(plot_path, facecolor='white', bbox_inches='tight')
plt.show()
print(f'📈 Chart successfully exported to: {plot_path}')